# Análisis exploratorio de datos

In [33]:
# Importaciones

import pandas as pd
import pyarrow as pa

from pathlib import Path
from collections import Counter

In [2]:
# Rutas

RUTAS = {
    "OGLE-III" : Path("../data/raw/curves/OGLE-III"),
    "OGLE-IV" : Path("../data/raw/curves/OGLE-IV")
}

In [5]:
ESPERADOS = {
    "OGLE-III": 44_217,
    "OGLE-IV": 128_472,
}

In [3]:
# Inventariar archivos

def inventariar_archivos(ruta):
    archivos = [archivo for archivo in ruta.rglob("*") if archivo.is_file()]

    extensiones = Counter(
        archivo.suffix.lower() if archivo.suffix else "[sin extensión]"
        for archivo in archivos
    )

    return {
        "ruta_existe": ruta.exists(),
        "total_archivos": len(archivos),
        "archivos_vacios": sum(archivo.stat().st_size == 0 for archivo in archivos),
        "extensiones": extensiones,
    }


for proyecto, ruta in RUTAS.items():
    resultado = inventariar_archivos(ruta)

    print(f"\n{proyecto}")
    print(f"Ruta: {ruta}")
    print(f"Existe: {resultado['ruta_existe']}")
    print(f"Total de archivos: {resultado['total_archivos']:,}")
    print(f"Archivos vacíos: {resultado['archivos_vacios']:,}")
    print("Extensiones:")

    for extension, cantidad in resultado["extensiones"].most_common():
        print(f"  {extension}: {cantidad:,}")


OGLE-III
Ruta: ..\data\raw\curves\OGLE-III
Existe: True
Total de archivos: 87,168
Archivos vacíos: 0
Extensiones:
  .dat: 87,168

OGLE-IV
Ruta: ..\data\raw\curves\OGLE-IV
Existe: True
Total de archivos: 209,932
Archivos vacíos: 0
Extensiones:
  .dat: 209,932


In [6]:
# Identificar curvas

def identificar_curvas(ruta):
    archivos_dat = [
        archivo
        for archivo in ruta.rglob("*")
        if archivo.is_file() and archivo.suffix.lower() == ".dat"
    ]

    curvas = [
        archivo
        for archivo in archivos_dat
        if "RRLYR" in archivo.stem.upper()
    ]

    ids = [archivo.stem for archivo in curvas]

    return {
        "archivos_dat": archivos_dat,
        "curvas": curvas,
        "ids": ids,
    }


resumen_curvas = {}

for proyecto, ruta in RUTAS.items():
    resultado = identificar_curvas(ruta)
    resumen_curvas[proyecto] = resultado

    ids_unicos = set(resultado["ids"])
    duplicados = len(resultado["ids"]) - len(ids_unicos)
    diferencia = ESPERADOS[proyecto] - len(ids_unicos)

    print(f"\n{proyecto}")
    print(f"Total de archivos .dat: {len(resultado['archivos_dat']):,}")
    print(f"Archivos candidatos a curvas: {len(resultado['curvas']):,}")
    print(f"Identificadores únicos: {len(ids_unicos):,}")
    print(f"Archivos duplicados por identificador: {duplicados:,}")
    print(f"Total publicado: {ESPERADOS[proyecto]:,}")
    print(f"Diferencia inicial: {diferencia:,}")


OGLE-III
Total de archivos .dat: 87,168
Archivos candidatos a curvas: 87,168
Identificadores únicos: 44,217
Archivos duplicados por identificador: 42,951
Total publicado: 44,217
Diferencia inicial: 0

OGLE-IV
Total de archivos .dat: 209,932
Archivos candidatos a curvas: 209,932
Identificadores únicos: 126,958
Archivos duplicados por identificador: 82,974
Total publicado: 128,472
Diferencia inicial: 1,514


In [7]:
proyecto = "OGLE-IV"
resultado = resumen_curvas[proyecto]

frecuencia_por_id = Counter(resultado["ids"])
distribucion_frecuencias = Counter(frecuencia_por_id.values())

frecuencia_por_carpeta = Counter(
    str(archivo.parent.relative_to(RUTAS[proyecto]))
    for archivo in resultado["curvas"]
)

print("Cantidad de archivos por identificador:")

for cantidad_archivos, cantidad_ids in sorted(distribucion_frecuencias.items()):
    print(
        f"  {cantidad_archivos} archivo(s): "
        f"{cantidad_ids:,} identificadores"
    )

print("\nCarpetas con mayor cantidad de curvas:")

for carpeta, cantidad in frecuencia_por_carpeta.most_common(20):
    print(f"  {carpeta}: {cantidad:,}")

Cantidad de archivos por identificador:
  1 archivo(s): 43,984 identificadores
  2 archivo(s): 82,974 identificadores

Carpetas con mayor cantidad de curvas:
  query_1786855024.80031a\I: 48,032
  query_1786853816.92220t\I: 41,209
  query_1786853816.92220t\V: 38,602
  query_1786855024.80031a\V: 22,403
  query_1786856050.11128v\I: 18,912
  query_1786854744.50090l\I: 11,652
  query_1786856050.11128v\V: 10,139
  query_1786853168.49485w\I: 6,823
  query_1786853168.49485w\V: 6,503
  query_1786854744.50090l\V: 5,188
  query_1786856654.19554r\I: 330
  query_1786856654.19554r\V: 139


## Cruce con archivos Ident.dat

In [13]:
archivos_ident = sorted(
    archivo
    for archivo in Path("../data/raw/ident").rglob("*.dat")
    if archivo.name.lower().startswith("ident-")
)

print(f"Archivos de identificación encontrados: {len(archivos_ident)}\n")

for archivo in archivos_ident:
    print(f"ARCHIVO: {archivo}")
    
    with archivo.open("r", encoding="utf-8") as contenido:
        for _ in range(3):
            linea = contenido.readline().rstrip()
            print(repr(linea))
    
    print()

Archivos de identificación encontrados: 7

ARCHIVO: ..\data\raw\ident\ident-BLG-OGLE-III.dat
'OGLE-BLG-RRLYR-00001  BLG366.6  56455 RRab  17:05:07.49 -32:37:57.2'
'OGLE-BLG-RRLYR-00002  BLG366.8  59262 RRc   17:05:08.09 -32:56:38.1'
'OGLE-BLG-RRLYR-00003  BLG366.6  56524 RRab  17:05:09.88 -32:39:52.8'

ARCHIVO: ..\data\raw\ident\ident-BLG-OGLE-IV.dat
'OGLE-BLG-RRLYR-00001  RRab  17:05:07.49 -32:37:57.2  BLG897.03.53266  BLG366.6.56455'
'OGLE-BLG-RRLYR-00002  RRc   17:05:08.09 -32:56:38.1  BLG898.28.59490  BLG366.8.59262'
'OGLE-BLG-RRLYR-00003  RRab  17:05:09.88 -32:39:52.8  BLG897.03.53856  BLG366.6.56524'

ARCHIVO: ..\data\raw\ident\ident-GD-OGLE-IV.dat
'OGLE-GD-RRLYR-00001  RRab  10:35:56.15 -62:38:40.7  GD2047.21.18203  CAR116.6.22422'
'OGLE-GD-RRLYR-00002  RRab  10:36:12.01 -62:34:50.0  GD2047.21.17000  CAR116.5.6364'
'OGLE-GD-RRLYR-00003  RRab  10:36:26.74 -62:31:08.4  GD2047.29.844    CAR116.5.18588'

ARCHIVO: ..\data\raw\ident\ident-LMC-OGLE-III.dat
'OGLE-LMC-RRLYR-00001  LMC158

In [14]:
filas_comparacion = []

for archivo in archivos_ident:
    partes_nombre = archivo.stem.split("-")
    region = partes_nombre[1]
    proyecto = f"{partes_nombre[2]}-{partes_nombre[3]}"

    with archivo.open("r", encoding="utf-8") as contenido:
        ids_ident = [
            linea.split()[0]
            for linea in contenido
            if linea.strip() and not linea.lstrip().startswith("#")
        ]

    conjunto_ident = set(ids_ident)

    prefijo_region = f"OGLE-{region}-RRLYR-"
    conjunto_curvas = {
        identificador
        for identificador in resumen_curvas[proyecto]["ids"]
        if identificador.startswith(prefijo_region)
    }

    filas_comparacion.append({
        "proyecto": proyecto,
        "region": region,
        "filas_ident": len(ids_ident),
        "ids_ident_unicos": len(conjunto_ident),
        "duplicados_ident": len(ids_ident) - len(conjunto_ident),
        "ids_con_curva": len(conjunto_curvas),
        "ident_sin_curva": len(conjunto_ident - conjunto_curvas),
        "curva_sin_ident": len(conjunto_curvas - conjunto_ident),
    })

comparacion_ident_curvas = (
    pd.DataFrame(filas_comparacion)
    .sort_values(["proyecto", "region"])
    .reset_index(drop=True)
)

comparacion_ident_curvas

,proyecto,region,filas_ident,ids_ident_unicos,duplicados_ident,ids_con_curva,ident_sin_curva,curva_sin_ident
0,OGLE-III,BLG,16836,16836,0,16836,0,0
1,OGLE-III,LMC,24906,24906,0,24906,0,0
2,OGLE-III,SMC,2475,2475,0,2475,0,0
3,OGLE-IV,BLG,68509,68509,0,67274,1235,0
4,OGLE-IV,GD,11658,11658,0,11652,6,0
5,OGLE-IV,LMC,41471,41471,0,41209,262,0
6,OGLE-IV,SMC,6835,6835,0,6823,12,0


In [16]:
lineas_atipicas = []
distribucion_columnas = Counter()

for archivo in archivos_ident:
    if "OGLE-IV" not in archivo.stem:
        continue

    with archivo.open("r", encoding="utf-8") as contenido:
        for numero_linea, linea in enumerate(contenido, start=1):
            if not linea.strip() or linea.lstrip().startswith("#"):
                continue

            campos = linea.split()
            distribucion_columnas[len(campos)] += 1

            if len(campos) != 6:
                lineas_atipicas.append({
                    "archivo": archivo.name,
                    "linea": numero_linea,
                    "n_columnas": len(campos),
                    "contenido": repr(linea.rstrip()),
                })

print("Distribución del número de columnas:")
for n_columnas, cantidad in sorted(distribucion_columnas.items()):
    print(f"  {n_columnas} columnas: {cantidad:,} líneas")

print(f"\nLíneas con estructura distinta de 6 columnas: {len(lineas_atipicas):,}\n")

for registro in lineas_atipicas[:30]:
    print(registro)

Distribución del número de columnas:
  4 columnas: 7 líneas
  5 columnas: 77,768 líneas
  6 columnas: 32,872 líneas
  7 columnas: 15,413 líneas
  8 columnas: 2,208 líneas
  9 columnas: 205 líneas

Líneas con estructura distinta de 6 columnas: 95,601

{'archivo': 'ident-BLG-OGLE-IV.dat', 'linea': 8, 'n_columnas': 5, 'contenido': "'OGLE-BLG-RRLYR-00008  RRab  17:05:44.77 -32:29:35.2                   BLG366.5.65886'"}
{'archivo': 'ident-BLG-OGLE-IV.dat', 'linea': 16, 'n_columnas': 5, 'contenido': "'OGLE-BLG-RRLYR-00016  RRab  17:06:43.95 -32:29:58.7                   BLG366.4.65766'"}
{'archivo': 'ident-BLG-OGLE-IV.dat', 'linea': 103, 'n_columnas': 5, 'contenido': "'OGLE-BLG-RRLYR-00103  RRab  17:10:16.20 -31:53:53.2                   BLG365.3.78364'"}
{'archivo': 'ident-BLG-OGLE-IV.dat', 'linea': 107, 'n_columnas': 5, 'contenido': "'OGLE-BLG-RRLYR-00107  RRc   17:10:25.57 -31:53:58.3                   BLG365.3.82590'"}
{'archivo': 'ident-BLG-OGLE-IV.dat', 'linea': 122, 'n_columnas': 5, 

In [17]:
registros_ogle_iv = []

for archivo in archivos_ident:
    if "OGLE-IV" not in archivo.stem:
        continue

    region = archivo.stem.split("-")[1]

    with archivo.open("r", encoding="utf-8") as contenido:
        for linea in contenido:
            if not linea.strip() or linea.lstrip().startswith("#"):
                continue

            campos = linea.split()

            registros_ogle_iv.append({
                "region": region,
                "star_id": campos[0],
                "subtipo": campos[1],
                "ra": campos[2],
                "dec": campos[3],
            })

ident_ogle_iv = pd.DataFrame(registros_ogle_iv)

ids_curvas_ogle_iv = set(resumen_curvas["OGLE-IV"]["ids"])

faltantes_ogle_iv = ident_ogle_iv[
    ~ident_ogle_iv["star_id"].isin(ids_curvas_ogle_iv)
].copy()

print(f"Registros totales en los ident: {len(ident_ogle_iv):,}")
print(f"Estrellas sin ninguna curva: {len(faltantes_ogle_iv):,}\n")

print("Estrellas sin curva por región y subtipo:")

pd.crosstab(
    faltantes_ogle_iv["region"],
    faltantes_ogle_iv["subtipo"],
    margins=True
)

Registros totales en los ident: 128,473
Estrellas sin ninguna curva: 1,515

Estrellas sin curva por región y subtipo:


subtipo,RRab,RRc,RRd,aRRd,All
region,,,,,
BLG,827,355,4,49,1235
GD,6,0,0,0,6
LMC,170,82,10,0,262
SMC,8,3,1,0,12
All,1011,440,15,49,1515


In [20]:
tabla_faltantes = pd.crosstab(
    faltantes_ogle_iv["region"],
    faltantes_ogle_iv["subtipo"],
    margins=True
)

print(tabla_faltantes.to_string())

subtipo  RRab  RRc  RRd  aRRd   All
region                             
BLG       827  355    4    49  1235
GD          6    0    0     0     6
LMC       170   82   10     0   262
SMC         8    3    1     0    12
All      1011  440   15    49  1515


## Cruce con archivos de períodos

In [22]:
ruta_periodos = Path("../data/raw/periods")

archivos_periodos = sorted(
    archivo
    for archivo in ruta_periodos.glob("*.dat")
    if archivo.is_file()
)

print(f"Archivos de períodos encontrados: {len(archivos_periodos)}\n")

for archivo in archivos_periodos:
    print(f"ARCHIVO: {archivo.name}")

    with archivo.open("r", encoding="utf-8") as contenido:
        primera_linea = contenido.readline().rstrip()

    print(f"PRIMERA LÍNEA: {primera_linea!r}\n")

Archivos de períodos encontrados: 28

ARCHIVO: OGLE-III-BLG-RRab.dat
PRIMERA LÍNEA: 'OGLE-BLG-RRLYR-00001  15.545 16.817  0.73258757 0.00000240  5000.58187  0.438  0.435 4.836  0.253 3.590'

ARCHIVO: OGLE-III-BLG-RRc.dat
PRIMERA LÍNEA: 'OGLE-BLG-RRLYR-00002  15.843 16.799  0.30153381 0.00000067  5000.05809  0.286  0.155 4.957  0.093 3.897'

ARCHIVO: OGLE-III-BLG-RRd.dat
PRIMERA LÍNEA: 'OGLE-BLG-RRLYR-00039  15.927 17.414  0.35827941 0.00000143  5000.16981  0.266  0.136 4.955  0.104 3.903  0.48137703 0.00000679  5000.16834  0.098  0.112 3.921  0.000   -'

ARCHIVO: OGLE-III-GD-rrlyr.dat
PRIMERA LÍNEA: 'OGLE-GD-RRLYR-0001  17.264 17.860 19.022  1.429 17.593  0.55556203 0.00000060  35'

ARCHIVO: OGLE-III-LMC-RRab.dat
PRIMERA LÍNEA: 'OGLE-LMC-RRLYR-00001 18.798 19.391  0.6347522 0.0000016  2184.64705  0.508  0.510 4.358  0.280 2.718'

ARCHIVO: OGLE-III-LMC-RRc.dat
PRIMERA LÍNEA: 'OGLE-LMC-RRLYR-00015 18.833 19.353  0.3557413 0.0000011  2171.69751  0.253  0.000   -    0.000   -'

ARCHIVO: OG

In [23]:
ids_periodos_por_grupo = {}
filas_periodos_por_grupo = Counter()

for archivo in archivos_periodos:
    partes = archivo.stem.split("-")
    proyecto = f"{partes[0]}-{partes[1]}"
    region = partes[2]
    grupo = (proyecto, region)

    ids_archivo = []

    with archivo.open("r", encoding="utf-8") as contenido:
        for linea in contenido:
            if not linea.strip() or linea.lstrip().startswith("#"):
                continue

            ids_archivo.append(linea.split()[0])

    ids_periodos_por_grupo.setdefault(grupo, []).extend(ids_archivo)
    filas_periodos_por_grupo[grupo] += len(ids_archivo)

filas_cobertura = []

for grupo, ids_periodos in sorted(ids_periodos_por_grupo.items()):
    proyecto, region = grupo

    conjunto_periodos = set(ids_periodos)

    if proyecto in resumen_curvas:
        prefijo = f"OGLE-{region}-RRLYR-"
        conjunto_curvas = {
            identificador
            for identificador in resumen_curvas[proyecto]["ids"]
            if identificador.startswith(prefijo)
        }
    else:
        conjunto_curvas = set()

    filas_cobertura.append({
        "proyecto": proyecto,
        "region": region,
        "filas_periodos": len(ids_periodos),
        "ids_periodo_unicos": len(conjunto_periodos),
        "duplicados_periodos": len(ids_periodos) - len(conjunto_periodos),
        "ids_con_curva": len(conjunto_curvas),
        "curvas_con_periodo": len(conjunto_curvas & conjunto_periodos),
        "curvas_sin_periodo": len(conjunto_curvas - conjunto_periodos),
        "periodos_sin_curva": len(conjunto_periodos - conjunto_curvas),
    })

cobertura_periodos = pd.DataFrame(filas_cobertura)

print(cobertura_periodos.to_string(index=False))

proyecto region  filas_periodos  ids_periodo_unicos  duplicados_periodos  ids_con_curva  curvas_con_periodo  curvas_sin_periodo  periodos_sin_curva
OGLE-III    BLG           16836               16836                    0          16836               16836                   0                   0
OGLE-III     GD              45                  45                    0              0                   0                   0                  45
OGLE-III    LMC           24906               24906                    0          24906               24906                   0                   0
OGLE-III    SMC            2475                2475                    0           2475                2475                   0                   0
 OGLE-IV    BLG           68509               68509                    0          67274               67274                   0                1235
 OGLE-IV     GD           11658               11658                    0          11652               11652     

### Conteo de mediciones por curva

In [24]:
registros_nobs = []

for proyecto in ["OGLE-III", "OGLE-IV"]:
    curvas_i = [
        archivo
        for archivo in resumen_curvas[proyecto]["curvas"]
        if archivo.parent.name.upper() == "I"
    ]

    for numero, archivo in enumerate(curvas_i, start=1):
        with archivo.open("r", encoding="utf-8") as contenido:
            n_obs = sum(1 for linea in contenido if linea.strip())

        registros_nobs.append({
            "proyecto": proyecto,
            "region": archivo.stem.split("-")[1],
            "star_id": archivo.stem,
            "n_obs": n_obs,
            "ruta_curva": archivo,
        })

        if numero % 20_000 == 0:
            print(
                f"{proyecto}: "
                f"{numero:,} de {len(curvas_i):,} curvas procesadas"
            )

conteos_observaciones = pd.DataFrame(registros_nobs)

resumen_nobs = (
    conteos_observaciones
    .groupby(["proyecto", "region"])["n_obs"]
    .agg(
        total_curvas="size",
        minimo="min",
        mediana="median",
        maximo="max",
        con_300_o_mas=lambda serie: (serie >= 300).sum(),
        con_menos_de_300=lambda serie: (serie < 300).sum(),
    )
    .reset_index()
)

print("\nResumen de observaciones en banda I:\n")
print(resumen_nobs.to_string(index=False))

OGLE-III: 20,000 de 44,217 curvas procesadas
OGLE-III: 40,000 de 44,217 curvas procesadas
OGLE-IV: 20,000 de 126,958 curvas procesadas
OGLE-IV: 40,000 de 126,958 curvas procesadas
OGLE-IV: 60,000 de 126,958 curvas procesadas
OGLE-IV: 80,000 de 126,958 curvas procesadas
OGLE-IV: 100,000 de 126,958 curvas procesadas
OGLE-IV: 120,000 de 126,958 curvas procesadas

Resumen de observaciones en banda I:

proyecto region  total_curvas  minimo  mediana  maximo  con_300_o_mas  con_menos_de_300
OGLE-III    BLG         16836      11    623.0    5173          11061              5775
OGLE-III    LMC         24906      24    381.0    1788          23995               911
OGLE-III    SMC          2475      89    696.0    2216           2448                27
 OGLE-IV    BLG         67274      10    163.0   15016          30824             36450
 OGLE-IV     GD         11652      12    123.0    1202              4             11648
 OGLE-IV    LMC         41209       9    477.0     941          30581  

In [25]:
ejemplos_curvas = (
    conteos_observaciones
    .sort_values(["proyecto", "region", "star_id"])
    .groupby(["proyecto", "region"], as_index=False)
    .first()
)

for _, fila in ejemplos_curvas.iterrows():
    print(
        f"\n{fila['proyecto']} | {fila['region']} | "
        f"{fila['star_id']} | n_obs={fila['n_obs']}"
    )

    with fila["ruta_curva"].open("r", encoding="utf-8") as contenido:
        for _ in range(5):
            print(repr(contenido.readline().rstrip()))


OGLE-III | BLG | OGLE-BLG-RRLYR-00001 | n_obs=95
'2123.69145 15.378 0.004'
'2128.48357 15.602 0.005'
'2131.48397 15.649 0.005'
'2134.51581 15.755 0.005'
'2137.53362 15.663 0.005'

OGLE-III | LMC | OGLE-LMC-RRLYR-00001 | n_obs=332
'2184.81907 18.725 0.060'
'2188.86966 18.878 0.075'
'2191.78070 18.716 0.040'
'2200.80028 18.808 0.045'
'2206.73541 19.026 0.088'

OGLE-III | SMC | OGLE-SMC-RRLYR-0001 | n_obs=654
'2088.91535 18.868 0.078'
'2090.87535 19.540 0.165'
'2103.92958 18.704 0.040'
'2104.94812 19.291 0.098'
'2106.84596 18.910 0.066'

OGLE-IV | BLG | OGLE-BLG-RRLYR-00001 | n_obs=121
'6785.90808 15.411 0.007'
'6789.68669 15.479 0.008'
'6790.71714 15.691 0.009'
'6791.72986 15.404 0.008'
'6796.73450 15.826 0.009'

OGLE-IV | GD | OGLE-GD-RRLYR-00001 | n_obs=99
'7787.78742 17.827 0.028'
'7790.80061 17.546 0.021'
'7795.73967 17.398 0.026'
'7796.79352 17.252 0.023'
'7797.72861 17.899 0.038'

OGLE-IV | LMC | OGLE-LMC-RRLYR-00001 | n_obs=403
'5262.50439 18.634 0.051'
'5264.50622 18.587 0.040'


In [27]:
curvas_seleccionadas = (
    conteos_observaciones[
        conteos_observaciones["n_obs"] >= 300
    ]
    .copy()
    .reset_index(drop=True)
)

print(f"Curvas seleccionadas: {len(curvas_seleccionadas):,}")

print(
    curvas_seleccionadas
    .groupby(["proyecto", "region"])
    .size()
    .rename("cantidad")
    .to_string()
)

Curvas seleccionadas: 104,008
proyecto  region
OGLE-III  BLG       11061
          LMC       23995
          SMC        2448
OGLE-IV   BLG       30824
          GD            4
          LMC       30581
          SMC        5095


In [28]:
registros_validacion = []

for numero, fila in curvas_seleccionadas.iterrows():
    total_filas = 0
    filas_validas = 0
    formato_invalido = 0
    valores_no_numericos = 0
    error_no_positivo = 0

    with fila["ruta_curva"].open("r", encoding="utf-8") as contenido:
        for linea in contenido:
            if not linea.strip():
                continue

            total_filas += 1
            campos = linea.split()

            if len(campos) != 3:
                formato_invalido += 1
                continue

            try:
                tiempo, magnitud, error = map(float, campos)
            except ValueError:
                valores_no_numericos += 1
                continue

            valores_finitos = all(
                -float("inf") < valor < float("inf")
                for valor in (tiempo, magnitud, error)
            )

            if not valores_finitos:
                valores_no_numericos += 1
                continue

            if error <= 0:
                error_no_positivo += 1
                continue

            filas_validas += 1

    registros_validacion.append({
        **fila.to_dict(),
        "total_filas": total_filas,
        "n_obs_validas": filas_validas,
        "formato_invalido": formato_invalido,
        "valores_no_numericos": valores_no_numericos,
        "error_no_positivo": error_no_positivo,
    })

    if (numero + 1) % 20_000 == 0:
        print(
            f"{numero + 1:,} de "
            f"{len(curvas_seleccionadas):,} curvas validadas"
        )

validacion_curvas = pd.DataFrame(registros_validacion)

resumen_validacion = (
    validacion_curvas
    .assign(
        conserva_umbral=validacion_curvas["n_obs_validas"] >= 300,
        filas_invalidas=(
            validacion_curvas["formato_invalido"]
            + validacion_curvas["valores_no_numericos"]
            + validacion_curvas["error_no_positivo"]
        ),
    )
    .groupby(["proyecto", "region"])
    .agg(
        curvas_candidatas=("star_id", "size"),
        curvas_con_300_validas=("conserva_umbral", "sum"),
        filas_invalidas=("filas_invalidas", "sum"),
    )
    .reset_index()
)

print("\nResumen de validación:\n")
print(resumen_validacion.to_string(index=False))

20,000 de 104,008 curvas validadas
40,000 de 104,008 curvas validadas
60,000 de 104,008 curvas validadas
80,000 de 104,008 curvas validadas
100,000 de 104,008 curvas validadas

Resumen de validación:

proyecto region  curvas_candidatas  curvas_con_300_validas  filas_invalidas
OGLE-III    BLG              11061                   11061                0
OGLE-III    LMC              23995                   23995                0
OGLE-III    SMC               2448                    2448                0
 OGLE-IV    BLG              30824                   30824                0
 OGLE-IV     GD                  4                       4                0
 OGLE-IV    LMC              30581                   30581                0
 OGLE-IV    SMC               5095                    5095                0


In [29]:
registros_periodos = []
incidencias_periodos = []

for archivo in archivos_periodos:
    partes_nombre = archivo.stem.split("-")
    proyecto = f"{partes_nombre[0]}-{partes_nombre[1]}"
    region = partes_nombre[2]
    subtipo_archivo = partes_nombre[3]

    with archivo.open("r", encoding="utf-8") as contenido:
        for numero_linea, linea in enumerate(contenido, start=1):
            if not linea.strip() or linea.lstrip().startswith("#"):
                continue

            campos = linea.split()

            try:
                star_id = campos[0]
                periodo_principal = float(campos[3])

                if subtipo_archivo.lower() in {"rrd", "arrd"}:
                    periodo_secundario = float(campos[11])
                else:
                    periodo_secundario = None

            except (IndexError, ValueError):
                incidencias_periodos.append({
                    "archivo": archivo.name,
                    "linea": numero_linea,
                    "contenido": repr(linea.rstrip()),
                })
                continue

            registros_periodos.append({
                "proyecto": proyecto,
                "region": region,
                "star_id": star_id,
                "subtipo": subtipo_archivo,
                "periodo_principal": periodo_principal,
                "periodo_secundario": periodo_secundario,
                "archivo_periodos": archivo.name,
            })

catalogo_periodos = pd.DataFrame(registros_periodos)

duplicados_periodos = catalogo_periodos.duplicated(
    subset=["proyecto", "star_id"],
    keep=False,
)

print(f"Registros de períodos: {len(catalogo_periodos):,}")
print(f"Incidencias de lectura: {len(incidencias_periodos):,}")
print(
    "Registros duplicados por proyecto y star_id: "
    f"{duplicados_periodos.sum():,}\n"
)

print(
    catalogo_periodos
    .groupby(["proyecto", "region", "subtipo"])
    .size()
    .rename("cantidad")
    .to_string()
)

Registros de períodos: 172,735
Incidencias de lectura: 0
Registros duplicados por proyecto y star_id: 0

proyecto  region  subtipo
OGLE-III  BLG     RRab       11756
                  RRc         4989
                  RRd           91
          GD      rrlyr         45
          LMC     RRab       17693
                  RRc         4958
                  RRd          986
                  RRe         1269
          SMC     RRab        1933
                  RRc          175
                  RRd          258
                  RRe          109
OGLE-IV   BLG     RRab       48859
                  RRc        19267
                  RRd          334
                  aRRd          49
          GD      RRab        8880
                  RRc         2625
                  RRd          137
                  aRRd          16
          LMC     RRab       29151
                  RRc        10198
                  RRd         2100
                  aRRd          22
          SMC     RRab       

In [30]:
curvas_finales = (
    validacion_curvas[
        validacion_curvas["n_obs_validas"] >= 300
    ]
    .copy()
    .reset_index(drop=True)
)

columnas_periodos = [
    "proyecto",
    "region",
    "star_id",
    "subtipo",
    "periodo_principal",
    "periodo_secundario",
    "archivo_periodos",
]

manifiesto_curvas = curvas_finales.merge(
    catalogo_periodos[columnas_periodos],
    on=["proyecto", "region", "star_id"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

periodos_principales_validos = (
    manifiesto_curvas["periodo_principal"].notna()
    & (manifiesto_curvas["periodo_principal"] > 0)
)

requiere_periodo_secundario = (
    manifiesto_curvas["subtipo"]
    .str.lower()
    .isin(["rrd", "arrd"])
)

periodos_secundarios_faltantes = (
    requiere_periodo_secundario
    & manifiesto_curvas["periodo_secundario"].isna()
)

print(f"Curvas finales: {len(curvas_finales):,}")
print(f"Registros en el manifiesto: {len(manifiesto_curvas):,}\n")

print("Resultado del cruce:")
print(manifiesto_curvas["_merge"].value_counts().to_string())

print(
    "\nPeríodos principales válidos: "
    f"{periodos_principales_validos.sum():,}"
)
print(
    "Períodos principales inválidos o ausentes: "
    f"{(~periodos_principales_validos).sum():,}"
)
print(
    "RRd/aRRd sin período secundario: "
    f"{periodos_secundarios_faltantes.sum():,}"
)

Curvas finales: 104,008
Registros en el manifiesto: 104,008

Resultado del cruce:
_merge
both          104008
left_only          0
right_only         0

Períodos principales válidos: 104,008
Períodos principales inválidos o ausentes: 0
RRd/aRRd sin período secundario: 0


In [36]:
ruta_salida = Path(
    "../data/interim/manifiesto_curvas_validas.csv"
)

ruta_salida.parent.mkdir(parents=True, exist_ok=True)

manifiesto_exportar = (
    manifiesto_curvas
    .drop(columns="_merge")
    .copy()
)

manifiesto_exportar["ruta_curva"] = (
    manifiesto_exportar["ruta_curva"].astype(str)
)

manifiesto_exportar.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8",
)

manifiesto_verificacion = pd.read_csv(
    ruta_salida,
    encoding="utf-8",
)

claves_duplicadas = manifiesto_verificacion.duplicated(
    subset=["proyecto", "star_id"]
).sum()

print(f"Archivo exportado: {ruta_salida.resolve()}")
print(f"Filas guardadas: {len(manifiesto_verificacion):,}")
print(f"Columnas guardadas: {len(manifiesto_verificacion.columns)}")
print(f"Claves proyecto–star_id duplicadas: {claves_duplicadas:,}")
print(
    "Tamaño del archivo: "
    f"{ruta_salida.stat().st_size / 1024**2:,.2f} MB"
)

Archivo exportado: C:\Users\adanm\Code\tesis\data\interim\manifiesto_curvas_validas.csv
Filas guardadas: 104,008
Columnas guardadas: 14
Claves proyecto–star_id duplicadas: 0
Tamaño del archivo: 16.72 MB
